# 04. Adaptive RAG: 질문 유형에 따른 적응형 검색

> 모든 질문이 같은 RAG 파이프라인을 거칠 필요는 없어요. Query Router로 vectorstore · web_search · LLM 단독을 골라 보내는 적응형 RAG를 구현해요.

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

1. **Query Router**를 구현하여 질문을 `vectorstore` 또는 `web_search`로 자동 분류할 수 있어요
2. **5개 핵심 노드**(route_question, retrieve, grade_documents, generate, transform_query + web_search)로 구성된 Adaptive RAG 그래프를 만들 수 있어요
3. **3단계 조건부 라우팅**(질문 라우팅 → 문서 평가 → 환각/관련성 검증)으로 검색 전략을 동적으로 선택할 수 있어요
4. **Hallucination Grader**와 **Answer Grader**를 조합하여 생성된 답변의 품질을 자동 검증할 수 있어요
5. CRAG, Self-RAG와 Adaptive RAG의 차이를 설명하고, 언제 어떤 방식을 선택할지 판단할 수 있어요

## 사전 지식

- 이전 노트북 `03-CRAG-Self-RAG.ipynb`에서 배운 GradeDocuments, Hallucination Grader, Query Rewriter 패턴
- LangGraph의 StateGraph, 조건부 엣지, MemorySaver 기본 사용법
- Pydantic BaseModel과 `with_structured_output()`으로 구조화된 출력 다루기

## Adaptive RAG란?

**Adaptive RAG**는 [논문(arXiv:2403.14403)](https://arxiv.org/abs/2403.14403)에서 제안된 전략으로, 질문의 성격에 따라 다른 검색 전략을 선택해요.

> 🔑 **핵심 개념**: Adaptive RAG는 **대형병원의 접수 시스템**과 같아요. 환자가 들어오면 접수 창구(Query Router)에서 증상을 듣고 "내과로 가세요" 또는 "외과로 가세요"를 먼저 결정해요. CRAG/Self-RAG가 "일단 내과에 가봤다가 아니면 외과로 가는" 방식이라면, Adaptive RAG는 **처음부터 맞는 곳으로 안내**해요.

### 왜 이전 방식으로는 부족한가요?

CRAG와 Self-RAG는 항상 벡터스토어 검색부터 시작해요. 하지만 "2024년 노벨상 수상자"처럼 PDF에 없는 것이 명백한 질문도 일단 벡터 검색을 하고 → 실패하고 → 그제서야 웹 검색으로 가요. 이건 시간과 비용 낭비예요.

| 방식 | 특징 | 한계 |
|------|------|------|
| **Naive RAG** | 항상 벡터 검색 | 최신 정보 못 찾음 |
| **CRAG** | 문서 없으면 웹 검색으로 보강 | 처음부터 라우팅 없음 (시간 낭비) |
| **Self-RAG** | 생성 품질 자체 검증 | 웹 검색 없음 |
| **Adaptive RAG** | 질문 분류 → 최적 경로 선택 | 세 방식의 장점 결합 |

### Adaptive RAG 전체 아키텍처

```mermaid
flowchart TD
    A([START]) --> B{route_question<br/>질문 라우팅}
    B -- "vectorstore" --> C[retrieve<br/>벡터 검색]
    B -- "web_search" --> D[web_search<br/>웹 검색]
    D --> E[generate<br/>답변 생성]
    C --> F[grade_documents<br/>문서 관련성 평가]
    F --> G{decide_to_generate<br/>평가 결과?}
    G -- "관련 문서 있음" --> E
    G -- "관련 문서 없음" --> H[transform_query<br/>질문 재작성]
    H --> C
    E --> I{hallucination_check<br/>품질 검증}
    I -- "환각 감지" --> E
    I -- "질문 미해결" --> H
    I -- "좋은 답변" --> J([END])

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef decision fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef output fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
    classDef terminal fill:#f8d7da,stroke:#dc3545,color:#721c24

    class A,J terminal
    class C,D,E,H process
    class B,G,F,I decision
```

### Adaptive RAG = CRAG + Self-RAG + Query Router

| 구성 요소 | 출처 | 역할 |
|----------|------|------|
| **Query Router** | Adaptive RAG 고유 | 질문 → vectorstore/web_search 사전 분류 |
| **Retrieval Grader** | CRAG에서 가져옴 | 문서 관련성 평가 (yes/no) |
| **RAG Chain** | 공통 | 문서 기반 답변 생성 |
| **Hallucination Grader** | Self-RAG에서 가져옴 | 답변이 문서에 근거하는지 검증 |
| **Answer Grader** | Self-RAG에서 가져옴 | 답변이 질문을 해결하는지 검증 |
| **Query Rewriter** | CRAG/Self-RAG 공통 | 의미적 의도 추출, 검색 최적화 |

> 💡 **핵심 정리**: Adaptive RAG를 이해하면 이전 노트북의 모든 기법이 어떻게 하나로 합쳐지는지 볼 수 있어요. CRAG의 "웹 검색 보강" + Self-RAG의 "생성 품질 검증" + 새로운 "사전 라우팅"이 결합된 형태예요.

## 1. 환경 설정

In [ ]:
from dotenv import load_dotenv

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# ---------------------------------------------------
# LangSmith 추적 설정
# ---------------------------------------------------
# LangSmith를 사용하면 그래프 실행 과정을 시각적으로 추적할 수 있어요
import os

# os.environ["LANGCHAIN_TRACING_V2"] = "true"
# os.environ["LANGCHAIN_PROJECT"] = "LangGraph-RAG-Adaptive"

## 2. 기본 모델 및 Retriever 설정

Adaptive RAG는 5가지 평가기를 모두 사용해요. 각 평가기에 동일한 `gpt-4o-mini`를 재사용하되, `temperature=0`으로 고정하여 일관된 평가 결과를 얻어요.

> 💡 **실무 팁**: 평가기용 모델은 `temperature=0`이 필수예요. 평가 결과가 랜덤하게 바뀌면 시스템이 불안정해져요. 반면 답변 생성용 모델은 약간의 창의성을 위해 `temperature=0.1` 정도를 사용하기도 해요.

In [ ]:
# ---------------------------------------------------
# 기본 모델 초기화
# ---------------------------------------------------
from langchain.chat_models import init_chat_model

# 기본 모델: gpt-4o-mini (비용 효율, 학생 접근성)
# 다른 모델 옵션:
#   - "anthropic:claude-sonnet-4-5" (더 강력한 추론)
#   - "ollama:llama3" (로컬 실행)
llm = init_chat_model("openai:gpt-4o-mini", temperature=0)

# 기본 모델 준비 완료 (gpt-4o-mini, temperature=0)

In [ ]:
import os

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 3. Query Router 구성

Adaptive RAG에서 가장 중요한 첫 번째 단계예요. 질문을 분석하여 벡터스토어 검색과 웹 검색 중 어느 쪽이 적합한지 결정해요.

> 🔑 **핵심 개념**: `Literal["vectorstore", "web_search"]`로 타입을 제한하면 LLM이 이 두 값 중 하나만 반환해요. `with_structured_output(RouteQuery)`가 이를 강제하므로, 파싱 에러 없이 항상 유효한 라우팅 결정을 받아요.

> ⚠️ **자주 하는 실수**: 시스템 프롬프트에서 벡터스토어가 어떤 내용을 담고 있는지 명확히 알려줘야 해요. 라우터가 맥락 없이 판단하면 잘못된 경로로 가게 돼요.

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 테스트 1: 벡터스토어 문서에 있는 정보
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 4. 문서 관련성 평가기 (Retrieval Grader)

벡터스토어에서 검색된 문서가 실제로 질문과 관련이 있는지 검증해요. 이 단계에서 관련 없는 문서를 필터링하고, 관련 문서가 없으면 쿼리 재작성 경로로 보내요.

> 💡 **실무 팁**: 관련성 평가는 엄격하게 하지 않아도 돼요. "키워드나 의미적 유사성이 있으면 yes"라는 기준으로 설정하면, 불필요하게 쿼리를 재작성하는 빈도를 줄일 수 있어요.

In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 5. Hallucination Grader와 Answer Grader 구성

Self-RAG에서 배운 두 평가기를 그대로 사용해요. Adaptive RAG에서는 이 두 평가기를 조합하여 `hallucination_check` 조건부 엣지 하나로 관리해요.

> 💡 **핵심 정리**: 두 평가기가 순차적으로 동작하는 것을 설명해요:
> 1. **Hallucination Grader**: "이 답변이 문서에 근거하는가?" → No이면 바로 재생성
> 2. **Answer Grader**: "이 답변이 질문을 해결하는가?" → No이면 질문 재작성
> 두 평가를 모두 통과해야만 END로 진행해요.

In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 6. Query Rewriter와 Web Search Tool 구성

벡터 검색에 실패하거나 답변 품질이 낮을 때 사용하는 두 가지 도구예요.

In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 테스트: 간단한 질문이 어떻게 개선되는지 확인
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
from langchain_tavily import TavilySearch

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 테스트: 웹 검색 결과 확인
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 7. Adaptive RAG 그래프 구성

5개의 컴포넌트를 모두 준비했어요. 이제 LangGraph로 연결해볼게요.

### 그래프 상태(State) 정의

Adaptive RAG의 상태는 CRAG/Self-RAG보다 단순해요. `question`, `generation`, `documents` 세 가지만 있으면 돼요.

In [ ]:
from typing import Annotated, List
from typing_extensions import TypedDict

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


### 노드 함수 정의

5개의 실행 노드와 3개의 조건부 엣지 함수를 정의해요.

In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
from langchain_core.documents import Document

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


### 조건부 엣지 함수 정의

Adaptive RAG의 3단계 조건부 라우팅을 구현해요.

> 💡 **핵심 정리**: 세 개의 조건부 엣지가 서로 다른 시점에 다른 판단을 해요:
> - **route_question**: 처음 진입 시 소스 결정 (web vs vectorstore)
> - **decide_to_generate**: 문서 평가 후 진행 여부 결정 (generate vs rewrite)
> - **hallucination_check**: 생성 후 품질 결정 (end vs retry vs rewrite)

In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 8. 그래프 조립 및 컴파일

5개의 노드와 3개의 조건부 엣지를 연결하여 완전한 Adaptive RAG 그래프를 만들어요.

> 💡 **실무 팁**: `MemorySaver`를 체크포인터로 사용하면 대화 히스토리를 유지할 수 있어요. Adaptive RAG는 루프 구조가 있으므로 반드시 `recursion_limit`을 설정해야 해요.

In [ ]:
from langgraph.graph import END, StateGraph, START
from langgraph.checkpoint.memory import MemorySaver

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
from IPython.display import Image, display

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 그래프 흐름: START → route_question → (web_search | retrieve) → ... → generate → hallucination_check → (END | generate | transform_query)
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 9. Adaptive RAG 실행 테스트

두 가지 경로를 모두 테스트해요:
1. **벡터스토어 경로**: PDF에 있는 정보 질문
2. **웹 검색 경로**: 최신 정보 질문

In [ ]:
import uuid
from langchain_core.runnables import RunnableConfig

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 테스트 1: 벡터스토어 경로 (PDF에 있는 정보)
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
from langgraph.errors import GraphRecursionError

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 테스트 2: 웹 검색 경로 (최신 정보)
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 10. invoke()로 전체 실행 및 결과 확인

`stream()`이 아닌 `invoke()`를 사용하면 최종 상태만 반환해요. 중간 과정을 보지 않고 최종 답변만 원할 때 사용해요.

In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 11. 실습: 라우팅 전략 직접 테스트

다양한 질문으로 어떤 경로가 선택되는지 확인해볼게요.

> 💡 **핵심 정리**: 학생들이 각자 다른 질문을 입력하여 라우팅 결정이 어떻게 달라지는지 비교해보세요. 특히 "경계선" 질문 (2023년과 2024년의 경계에 있는 주제)이 어떻게 처리되는지 관찰해요.

In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 다양한 질문으로 라우팅 전략 테스트
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 12. 3가지 RAG 전략 비교

지금까지 배운 세 가지 전략을 정리해볼게요.

| 특성 | CRAG | Self-RAG | Adaptive RAG |
|------|------|----------|--------------|
| 검색 소스 | 벡터스토어 → 실패 시 웹 | 벡터스토어만 | 라우팅으로 사전 결정 |
| 문서 평가 | O (필터링) | O (필터링) | O (필터링) |
| 환각 검증 | X | O | O |
| 답변 관련성 | X | O | O |
| 자기 루프 | 1회 (재작성+웹검색) | 반복 가능 | 반복 가능 |
| 최신 정보 처리 | 사후 보완 | 불가 | 사전 라우팅 |
| 복잡도 | 중 | 중-상 | 상 |

### 프로덕션 환경에서의 선택 가이드

> 💡 **실무 팁**: 어떤 전략을 쓸지는 "LLM 호출 비용 vs 답변 품질" 트레이드오프로 결정해요.

| 시나리오 | 추천 전략 | 이유 |
|---------|----------|------|
| 사내 문서 Q&A (단일 소스) | **CRAG** | 웹 검색이 좋은 폴백이 됨 |
| 의료/법률 (높은 정확도 필수) | **Self-RAG** | 환각 검증이 핵심 |
| 복합 소스 (PDF + 웹 + DB) | **Adaptive RAG** | 사전 라우팅이 비용 절감 |
| MVP/프로토타입 | **Naive RAG** | 빠른 구현, 나중에 업그레이드 |
| 비용 민감한 서비스 | **CRAG** | LLM 호출 횟수가 적음 |

> 🔑 **핵심 개념**: Adaptive RAG는 CRAG와 Self-RAG의 장점을 결합해요:
> - CRAG의 "웹 검색 보강" + Self-RAG의 "생성 품질 검증" + 새로운 "사전 라우팅"
> - 세 전략 중 가장 강력하지만 LLM 호출 횟수가 가장 많아 비용도 높아요

> ⚠️ **자주 하는 실수**: 처음부터 Adaptive RAG를 적용하려 하지 마세요. Naive RAG → CRAG → Self-RAG → Adaptive RAG 순서로 점진적으로 개선하면서, LangSmith로 각 단계의 품질을 측정하는 것이 가장 효과적이에요.

## 핵심 요약

이 노트북에서 다음 내용을 배웠어요:

- **Query Router**: `Literal["vectorstore", "web_search"]`와 `with_structured_output(RouteQuery)`로 질문을 두 경로 중 하나로 자동 분류해요. 시스템 프롬프트에서 벡터스토어 내용을 명확히 설명하는 것이 핵심이에요.
- **3단계 조건부 라우팅**: (1) 최초 라우팅 → (2) 문서 평가 후 분기 → (3) 생성 품질 검증의 세 조건부 엣지가 순서대로 동작해요.
- **Hallucination Check**: `hallucination_grader`(환각 검증)와 `answer_grader`(관련성 검증)를 순차적으로 실행하는 복합 조건부 엣지예요. 두 검증을 모두 통과해야 END로 진행해요.
- **MemorySaver**: 루프 구조가 있는 그래프에서 상태를 체크포인트로 저장해요. `thread_id`로 여러 대화를 구분할 수 있어요.
- **Adaptive RAG vs CRAG/Self-RAG**: Adaptive RAG는 세 전략 중 가장 완전한 형태로, 웹 사전 라우팅 + 문서 평가 + 생성 품질 검증을 모두 포함해요.

## 다음 노트북 예고

다음 `05-Retrieval.ipynb`에서는 **검색 자체를 더 똑똑하게** 만드는 기법을 배워요. 2-Step / Agentic / Hybrid RAG 세 가지 아키텍처, **쿼리 강화(Query Enhancement)**로 N개의 다양한 쿼리를 생성하는 방법, **MMR(Maximal Marginal Relevance)**로 결과의 다양성을 보장하는 방법을 다뤄요. 지금까지 만든 RAG 루프의 "검색" 부분을 한 단계 업그레이드해요.